# Colab Dense Embedding & Indexing - LegalRAG

Notebook này dùng để **tạo Dense Vector Index (Embedding)** cho dữ liệu pháp luật lao động (Bộ luật Lao động 2019) sử dụng GPU (Tesla T4) trên Google Colab.

## Quy trình thực hiện:
1. **Runtime**: Đảm bảo đã chọn **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. **Setup**: Clone mã nguồn LegalRAG và cài đặt thư viện (`pip install -e .`).
3. **Build Index**: Sử dụng CLI `legal-rag index dense` để embed tập chunk được chọn.
4. **Kiểm tra**: Load index, kiểm tra shape embedding và chạy thử query mẫu.
5. **Tải kết quả**: Tải file `.pkl` về local hoặc lưu sang Google Drive.

In [ ]:
# 1. Kiểm tra GPU và môi trường PyTorch CUDA
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone repository LegalRAG và cài đặt package
!rm -rf /content/LegalRAG
!git clone https://github.com/ngminhduc129/LegalRAG.git /content/LegalRAG
%cd /content/LegalRAG
!pip install -e . -q

In [ ]:
# 3. Build Dense Index bằng sentence-transformers
# Mặc định model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
import os
from pathlib import Path

chunk_strategy = "legal_aware_run06"  # Tùy chọn: legal_aware_run06, fixed_300_run06, clause_run06, v.v.
input_chunks = f"data/chunks/{chunk_strategy}/labor_code_2019_legal_aware.jsonl"
output_index = f"data/indexes/dense/dense_{chunk_strategy}.pkl"

Path("data/indexes/dense").mkdir(parents=True, exist_ok=True)

print(f"Dang tao Dense Index cho {input_chunks}...")
!legal-rag index dense {input_chunks} --output {output_index} --model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 --batch-size 32 --max-length 256
print(f"\nHoan tat build index tai: {output_index}")

In [ ]:
# 4. Kiểm tra Index đã tạo và chạy thử Semantic Search với câu hỏi pháp lý mẫu
from pathlib import Path
from legal_rag.retrieval.dense import DenseIndex

index = DenseIndex.load(Path(output_index))
print(f"Total vectors: {len(index.documents)}")
print(f"Embeddings matrix shape: {index.embeddings.shape}")

# Chạy thử nghiệm 1 câu hỏi mẫu
query = "Người lao động có quyền đơn phương chấm dứt hợp đồng lao động khi nào?"
print(f"\nTest Query: {query}\n")
results = index.search(query, k=3)
for rank, (doc_id, score, doc) in enumerate(results, start=1):
    print(f"Rank {rank} (Score: {score:.4f}): [ID: {doc_id}] Điều {doc.get('article', 'N/A')} - {doc.get('article_title', '')}")
    print(f"  Snippet: {doc.get('content', '')[:160]}...\n")

In [ ]:
# 5. Tải file index về máy local (hoặc mount Google Drive để lưu trữ)
from google.colab import files

print(f"Tải file index {output_index} về máy...")
files.download(output_index)